In [1]:
import os
import datetime as dt
import pathlib
import requests
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# 1. Establish path and ensure it exists
DATA_RAW = pathlib.Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

# 2. Load environment variables
load_dotenv(dotenv_path="../.env")
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))

# 3. Class-specific file naming helper functions
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def safe_filename(prefix: str, meta: dict) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

# 4. Class-specific validation helper function
def validate_df(df: pd.DataFrame, required_cols: list, dtypes_map: dict) -> dict:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

Loaded ALPHAVANTAGE_API_KEY? True


In [2]:
SYMBOL = "AAPL"
use_alpha = bool(ALPHA_KEY) and ALPHA_KEY != "your_key_template"

if use_alpha:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        js = r.json()
        key = [k for k in js.keys() if "Time Series" in k]
        if not key:
            print("Alpha Vantage cap hit or premium wall. Falling back to yfinance.")
            use_alpha = False
        else:
            series = js[key[0]]
            df_api = pd.DataFrame(series).T.rename_axis('date').reset_index()
            df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
            df_api['date'] = pd.to_datetime(df_api['date'])
            df_api['close'] = pd.to_numeric(df_api['close'])
    except Exception as e:
        print(f"Alpha Vantage request failed: {e}. Falling back to yfinance.")
        use_alpha = False

if not use_alpha:
    # Fallback branch using yfinance
    df_api = yf.download(SYMBOL, period="6mo", interval="1d", auto_adjust=False, multi_level_index=False)
    df_api = df_api.reset_index()[['Date', 'Close']]
    df_api.columns = ['date', 'close']

# Sort data uniformly
df_api = df_api.sort_values('date').reset_index(drop=True)

# Execute structured validation function
api_validation = validate_df(
    df=df_api, 
    required_cols=['date', 'close'], 
    dtypes_map={'date': 'datetime64[ns]', 'close': 'float'}
)
print("API Validation Results:", api_validation)

# Generate filename and export
api_fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "ticker": SYMBOL})
api_out_path = DATA_RAW / api_fname
df_api.to_csv(api_out_path, index=False)
print("Successfully Ingested API Data:", api_out_path)

API Validation Results: {'na_total': 'Total NA values: 0'}
Successfully Ingested API Data: ..\data\raw\api_source-alpha_ticker-AAPL_20260817-212646.csv


In [3]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: student@nyu.edu)"}

print(f"Scraping table data from: {SCRAPE_URL}")
resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
table = soup.find('table', class_='wikitable') # Identifies the main component table

rows = []
for tr in table.find_all('tr'):
    cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
    if cells:
        rows.append(cells)

# Set up columns using first row headers
header, *data = rows
df_scrape = pd.DataFrame(data, columns=header)

# Run data type cleaning if 'Price' column exists
if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')

# Execute structured validation function
scrape_validation = validate_df(df=df_scrape, required_cols=list(df_scrape.columns), dtypes_map={})
print("Scraping Validation Results:", scrape_validation)

# Save to file path using custom template structure
scrape_fname = safe_filename(prefix="scrape", meta={"site": "wikipedia", "table": "djia"})
scrape_out_path = DATA_RAW / scrape_fname
df_scrape.to_csv(scrape_out_path, index=False)
print("Successfully Ingested Scraped Data:", scrape_out_path)

Scraping table data from: https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average
Scraping Validation Results: {'na_total': 'Total NA values: 0'}
Successfully Ingested Scraped Data: ..\data\raw\scrape_site-wikipedia_table-djia_20260817-212709.csv


## Ingestion Documentation & Context

### Data Strategy & Sources
1. **API Dataset**: Interacted with Alpha Vantage `TIME_SERIES_DAILY` with an automatic `yfinance` fallback engine capturing continuous 6-month transaction data for ticker symbol `AAPL`.
2. **Web Scraped Table**: Targeted the Dow Jones Industrial Average component index composition list found on Wikipedia (`https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average`), identifying the structural target via the `wikitable` element classification.

### Validation Checklists
* Extracted schemas are programmatically validated utilizing `validate_df()`. 
* Schema parameters mandate target tracking column visibility (`date` and `close`) along with zero missing data anomalies.

### Strategic Assumptions & Vulnerabilities
* **Assumption**: Upstream payload element formats are assumed to be static.
* **Risk Matrix**: External interface configurations represent point-of-failure risks. Structural variations within Wikipedia's table element layout or shifts behind API feature pricing tiers can disrupt ingestion flows.